# BTL_BigDeep — Deep Learning pipeline trên Kaggle

Chạy một lượt bằng **Save Version → Save & Run All**. Trước khi chạy:
1. **Add Data**: dataset CSE-CIC-IDS2018 (CSV) — đặt `RAW_DIR` ở ô dưới cho đúng thư mục `/kaggle/input/...`.
2. **Accelerator**: GPU (T4 x2 hoặc P100). **Internet**: On (cần cho git clone / pip / apt).
3. Chạy thử với `QUICK = True` (< 15 phút), xong mới đặt `QUICK = False`.

Kết quả: `/kaggle/working/kaggle_outputs.zip` (models/, results/, reports/, data/demo_test.parquet).

In [ ]:
# ===== Tham số =====
QUICK = False                      # True: 1 file CSV, train 50k / test 20k, 3 epoch
RAW_DIR = "/kaggle/input/ids-intrusion-csv"   # thư mục chứa CSV CSE-CIC-IDS2018 (tìm đệ quy)
WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/tcdat5/BTL_BIGDEEP.git"   # repo public: git clone
CODE_DATASET = ""                  # repo private: đường dẫn Kaggle Dataset chứa mã nguồn, vd "/kaggle/input/btl-bigdeep-code"
CODE_DIR = "/kaggle/working/BTL_BIGDEEP"
FILE_MAP = {}                      # nếu tên file khác config: {"Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv": "02-14-2018.csv"}

In [ ]:
# ===== 1. Thiết lập: phần cứng, mã nguồn, package =====
import os, sys, shutil, subprocess, importlib.util
print("CPU:", os.cpu_count(), "| RAM: %.1f GB" % (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "Không có GPU")
if CODE_DATASET:
    shutil.copytree(CODE_DATASET, CODE_DIR, dirs_exist_ok=True)
elif not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
need = {"torch": "torch", "onnx": "onnx", "onnxruntime": "onnxruntime", "sklearn": "scikit-learn",
        "lightgbm": "lightgbm", "pyarrow": "pyarrow", "matplotlib": "matplotlib", "yaml": "pyyaml",
        "scipy": "scipy", "pyspark": "pyspark==3.5.1"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
print("pip install:", missing or "không thiếu gì")

In [ ]:
# ===== 2. Java cho PySpark (Spark 3.5.1 hỗ trợ Java 17) =====
import glob
def java_ok():
    try:
        r = subprocess.run(["java", "-version"], capture_output=True, text=True)
        print((r.stderr or r.stdout).splitlines()[0]); return r.returncode == 0
    except FileNotFoundError:
        return False
if not java_ok():
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless", shell=True, check=True)
    java_ok()
homes = sorted(glob.glob("/usr/lib/jvm/java-17*"))
if homes and not os.environ.get("JAVA_HOME"):
    os.environ["JAVA_HOME"] = homes[0]
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))

In [ ]:
# ===== 3. Tìm dữ liệu =====
from train.kaggle_run_all import Runner
R = Runner(raw_dir=RAW_DIR, work_dir=WORK_DIR, quick=QUICK, file_map=FILE_MAP)
R.find_data()

In [ ]:
# ===== 4. Preprocess (Spark local[*]) — lỗi thì dừng =====
R.stage("preprocess")

In [ ]:
# ===== 5. EDA phần 1 (CPU) =====
R.stage("eda")

In [ ]:
# ===== 6a. Deep Autoencoder (GPU) — lỗi thì dừng =====
R.stage("ae")

In [ ]:
# ===== 6b. DNN classifier =====
R.stage("mlp")

In [ ]:
# ===== 6c. Variational AE =====
R.stage("vae")

In [ ]:
# ===== 6d. Denoising AE =====
R.stage("dae")

In [ ]:
# ===== 7. 6 baseline ML =====
R.stage("baselines")

In [ ]:
# ===== 8. Ablation + holdout (Bot, DDOS attack-HOIC, SSH-Bruteforce) =====
R.stage("ablation")
R.stage("holdout")

In [ ]:
# ===== 9. EDA latent + bảng/hình báo cáo =====
R.stage("eda_latent")
R.stage("report")

In [ ]:
# ===== 10. Đóng gói output =====
R.package()